# E9 Multistage Training

Author: Arush Arora

## Introduction
This codebase has mostly consisted of additive Graph Positional Encodings (GREPs) injections to provide nodal embeddings to the LLMs at hand. This new multi-stage training will rely on training an R-PEARL/GT to simply replicate the graph before expecting it to serve the LLM with **multiplicative** GREPs, which will be factored directly into the attention-mask matrix for rendition to the LLM (as a Hadamard product cover on the attention logits). Thus, the system will be more carefully trained to incorporate the variation in model architecture among GNNs and LLMs (in terms of their pre-trained weights rather than simply their mathematical foundations).

## Mathematical Overview

## The R-PEARL GNN
The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

### Graph Convolutional Network (GNN)
The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $\mathbf{P}(\cdot) = \mathbf{I}(\cdot)$, where $\mathbf{I}$ is the identity function):
$$\Phi(\mathbf{X}, \mathbf{S}, \mathcal{H}) = \mathbf{X}^{(L)}$$
$$\mathbf{X}^{(0)} = \mathbf{X} \qquad \mathbf{X}^{(l)} = \mathbf{P}\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \mathbf{S}^k\mathbf{X}^{(l - 1)}\mathbf{H}_k^{(l)}\Bigg)\Bigg]$$

#### Transformer
The Transformer architecture follows that of the Llama3.1-8B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$X = \tilde{X} + P$$

$${Z}_{1:t}^{(L)} = \operatorname{Trf}\bigg({X}_{1:t}, {\mathcal{T}}_l\bigg) \qquad {\mathcal{T}}_l = \begin{bmatrix}
{Q}_l & {K}_l & {V}_l & \left({W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big({Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

## Graph-Augmented LLM
The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$\renewcommand{\utilde}[1]{\underset{\sim}{#1}}$ $$\utilde{P} = \hat{\mathbb{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \utilde{S}, \mathcal{H}\Big)$$

$$\utilde{X} = \utilde{\tilde{X}} + \utilde{P}$$

$${\utilde{Z}}_{1:t}^{(L)} = \operatorname{Trf}\bigg({\utilde{X}}_{1:t}\bigg)$$

In [1]:
%env CUDA_VISIBLE_DEVICES=1

env: CUDA_VISIBLE_DEVICES=1


In [ ]:
# Import modules.
import torch
import random
import numpy as np
import sympy as sp
import networkx as nx

from prism.models import inference, gnn_llm, gt, loaders
from prism.eval import evaluate, loading
from prism.data import data

In [135]:
# Define a tensor rendering function.
def render_matrix(mat: torch.tensor, sig_figs: int = 0, decimals: int = 0):
    out = sp.Matrix(mat.detach().cpu().numpy())
    if sig_figs > 0:
        return sp.N(out, sig_figs)
    if decimals > 0:
        return out.applyfunc(lambda x: x.round(decimals))
    return out

In [3]:
# Standard options.
checkpoint = '../outputs/e8_new_base_models/e8_graph_mask_llm_gemma-4-12b-it_r16_4bit_6lbgcwc7/'
eval_path = '../data_store/revised/gen/nav100_n30_gemma_data/split/test_graphs'
include_edge_list = False
use_pretrained = True
device = 'cuda:0'

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Setup the Gemma 4 model from Hugging Face.
llm = AutoModelForCausalLM.from_pretrained("google/gemma-4-12B-it", dtype="auto", device_map="auto")

# Initialize a barebones/pretrained planner for testing.
if use_pretrained:
    model = gnn_llm.GraphMaskLLM(llm, use_edges=include_edge_list)
    tokenizer = AutoTokenizer.from_pretrained("google/gemma-4-12B-it")
else:
    model, tokenizer = loaders.graph_augmented_llm_from_pretrained(
        checkpoint, load_in_4bit=True, device=device,
    )

planner = inference.GraphAugmentedInMemoryLLM(model, tokenizer, include_edge_list)

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [5]:
from datasets import load_dataset

# Load in training dataset.
full_dataset = load_dataset("json", data_files=["../data/gen/nav100_n10_gemma_data/split/formatted_all_new_2turn__train.json"], split="train")
full_dataset = data.preprocess_dataset(
    full_dataset, tokenizer,
    architecture="graph_mask_llm",
    text_edge_list=include_edge_list,
)

In [6]:
# Setup eval infrastructure.
samples_by_graph, graph_file_by_name = loading.load_samples_by_graph(eval_path)
graph_file = random.choice(list(samples_by_graph.keys()))
eval_data = samples_by_graph[graph_file]
eval_data = {graph_file: [eval_data[random.randint(0, len(eval_data))]]}

In [7]:
print(f'/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/{graph_file}.html')

/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/data_gen_016.html


## Experiments

### §1 Testing a Pretrained `GraphMaskLLM`
The goal here is to test the model without any modifications to its internal architecture. By retrieving the necessary tensors for comparison, we can compute the perturbation of the attention logits and softmax distributions of the following matrix with $\mathbf{M}$ representing the block-matrix containing the graph adjacency $\mathbf{A} = \Big[\mathbb{I}\big((u, v) \in \mathcal{E}\big)\Big]_{u, v \in \mathcal{V}}$:
$$\mathbf{A}^{(h)}_l = \left[\frac{\exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1}\right) \odot M\right]}{\mathbf{1}^\top \exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1}\right) \odot M\right]}\right]^\top_{t \in [N]}$$

In [8]:
results = evaluate.eval_model_multiple_graphs(
    model, tokenizer, eval_data,
    include_edge_list=include_edge_list,
    use_icl=False,
    permutation=None,
    on_graph_done=None
)
results[graph_file].path_metrics

[spine-llm] client=GraphAugmentedInMemoryLLM, prompt_tokens=485
[spine-llm] graph_found=True, n_graphs=1, robot_location=bridge_1
[spine-llm] injection scope_start=255 / 485 tokens
[spine-llm] raw_output (first 500 chars): <think>
Relevant graph: bridge_1, mess_hall_1, crew_quarters_1, command_deck_1, comm_hub_1, observation_deck_1, captain_cabin_1, corridor_1, bio_lab_1, chem_lab_1, physics_lab_1, data_center_1, sample_vault_1, cryo_bay_1, specimen_room_1, research_hub_1, engine_room_1, reactor_core_1, power_grid_1, fuel_depot_1, maintenance_bay_1, ventilation_shaft_1, cargo_hold_1, airlock_1.

Reasoning:
1. Start at bridge_1.
2. Goal is cryo_bay_1.
3. Constraint 1: Must pass through chem_lab_1.
4. Constraint 2: 
{'primary_goal': '', 'relevant_graph': 'bridge_1, mess_hall_1, crew_quarters_1, command_deck_1, comm_hub_1, observation_deck_1, captain_cabin_1, corridor_1, bio_lab_1, chem_lab_1, physics_lab_1, data_center_1, sample_vault_1, cryo_bay_1, specimen_room_1, research_hub_1, engin

{'edge_validity_rate': 0.5,
 'nodes_exist_rate': 1.0,
 'full_path_valid_rate': 0.0,
 'start_goal_ok_rate': 1.0,
 'cost_optimality': None,
 'num_with_path': 1,
 'num_from_reasoning': 0,
 'num_rescued': 0,
 'structured_pass_rate': 0.0,
 'waypoints_ok_rate': 1.0,
 'avoid_ok_rate': 1.0,
 'required_edges_rate': 1.0,
 'num_structured': 1,
 'valid_path_rate': 0.0,
 'num_path_expected': 1,
 'hallucination_rate': 0.0}

### §2 Testing a Pretrained `GraphMaskLLM` with PE Injection into M (No Fintuning)
We would now like to instantiate a Graph Transformer and train it to replicate the graph adjacency matrix $\mathbf{A}$, defined above, to see if we achieve similar results.

In [ ]:
# Instantiate a Graph Transformer.
gt = gt.GraphTransformer(
    num_layers=3,
    pe_hidden_channels=256,
    pe_num_layers=2,
    d_model=1024,
    heads=8,
    num_samples=40,
    dropout=0.1,
    k_pe=2,
    k_gt=2,
    eps=1e-6,
    use_layer_norm=True
)

In [127]:
# Prepare a graph from the data to be used in the Graph Transformer.
from torch_geometric.utils import to_dense_adj
from prism.data import utils
import numpy as np
import sympy as sp

graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2])
adj_list = model._node_adjacency(graph, device=device).int()
adj = to_dense_adj(graph.edge_index).squeeze()
eigh_val, eigh_vec = torch.linalg.eigh(adj)
render_matrix(adj.int())

Matrix([
[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
[0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
[0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
[0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
[0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
[0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1,

In [128]:
render_matrix(eigh_vec, sig_figs=3)

Matrix([
[ -0.0185,    0.159,   0.0121,     0.003,  -0.166,     0.232,    0.025, -0.0619,   -0.135,        0,  -0.0218,   -0.126,    0.192,    0.115,  0.0844,   0.0351,    0.062,    -0.634,   0.539,   0.0887,  0.0734,  -0.157,  -0.203,  0.0475,  -0.0909,   0.076,  -0.0145, 0.0461],
[   -0.11,  -0.0392,   0.0158,     0.198, -0.0458,    0.0548,    -0.11,   0.092,   0.0468, -2.22e-8,   0.0604,  -0.0579,  -0.0168,   -0.252,  -0.594,    0.332,  -0.0695,   -0.0508,   0.195,   -0.381,  -0.335, -0.0132,   0.144,   0.235,  -0.0628, -0.0461,  -0.0134, 0.0335],
[  0.0913, -0.00783,  0.00327,   -0.0964, -0.0695,   -0.0222,   0.0606,  0.0527,    0.261, -3.89e-7,   -0.137,   -0.578,  -0.0811,  -0.0804,   -0.12,    0.164,  -0.0182,   -0.0238,  -0.164,      0.5, -0.0102,  -0.393,   0.236,  0.0757,  -0.0278, -0.0291, -0.00171, 0.0147],
[  -0.156,   0.0579,  -0.0228,   -0.0196,   0.166,   -0.0251,   0.0409,  -0.122,   -0.104, -1.04e-7,  -0.0756,   -0.236,  -0.0402,     0.11,   0.305,    0.493,    -0.26,

In [137]:
# Feed the matrix to the Graph Transformer.
render_matrix(gt(graph), 3)

Matrix([
[ -1.13, 0.991,  -0.608,   0.967,  -0.263, 0.424,   0.192, -1.44,  0.0591,  -1.78, -1.35, -1.72,   1.0, -0.886, -0.566,   0.421,  -1.21, 0.774, -0.714,  0.96, -0.0251,   -0.615,    -0.12,  1.58,   -0.03,   1.53, -0.357,  1.75,  1.48,    1.22,  1.88,  -1.02, -0.115,  -2.15, 0.733, -0.0303,  1.23, -0.367, -2.21,   0.0843,  1.38,  -0.304,  1.32,   0.315,  1.25,   0.788, -0.0558,  0.833, 1.24,    0.359,  0.852,  -1.01,  0.556, -0.533, 0.889, -0.781,  1.79, 0.643,    -1.1,  -1.31,  1.04,  0.933, 0.471,  -1.26,  -0.236, -0.656,  1.72,  -1.05, 2.56,  1.26, -0.0685, -0.645,  1.07,  -1.05, -0.559, -0.321,   0.894,   1.22, -0.179,     0.66,   -0.241, 0.567,  -1.58,   -0.8,  0.467,     1.18, -3.22,  -0.442,   -0.682,  -2.06,    0.222,   0.989,  -0.205,   -1.3,  -0.851,  0.0309, -0.594, -0.418,   0.759, -3.85,   1.7,  0.288,  -1.24,  1.51, -0.788,    0.76,  2.62,  1.24,   0.125, 0.854, 2.11, -0.794, -0.626,  0.545, -1.93, 0.981,   0.051,  -2.18, 0.0706,  -1.62,  -1.07,   1.6,  -1.16, 1.35

In [ ]:
# Train the Graph Transformer to reconstruct the eigenvectors of the graph adjacency.
optimizer = torch.optim.AdamW(gt.parameters(), lr=3e-4)

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss.
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # Results.
        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


train_loop()